## 환경 준비

아래 셀은 이 노트북에 필요한 Python 패키지가 설치되어 있는지 확인하고,
없으면 자동으로 설치한다. 이미 설치되어 있으면 빠르게 스킵된다.
터미널에서 미리 `uv sync`를 했다면 이 셀은 아무것도 설치하지 않는다.


In [ ]:
# === 의존성 자동 설치 (이미 설치되어 있으면 빠르게 스킵됨) ===
import subprocess, sys

_IMPORT_NAME_OVERRIDES = {
    "scikit-learn": "sklearn",
    "python-dateutil": "dateutil",
    "beautifulsoup4": "bs4",
}


def _ensure_packages(*packages):
    """누락된 패키지만 설치. 이미 있으면 스킵."""
    missing = []
    for pkg in packages:
        name = pkg.split(">=")[0].split("==")[0].split("[")[0]
        import_name = _IMPORT_NAME_OVERRIDES.get(name, name.replace("-", "_"))
        try:
            __import__(import_name)
        except ImportError:
            missing.append(pkg)
    if missing:
        print(f"Installing: {missing}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    else:
        print("All packages already installed ✓")

_ensure_packages(
    "pandas", "numpy", "matplotlib", "scikit-learn",
    "pydantic", "python-dateutil", "rich", "tqdm",
    "requests",
)

# 파인튜닝 전용 패키지 (설치 시간이 길 수 있음)
try:
    _ensure_packages(
        "torch", "transformers", "peft", "trl",
        "bitsandbytes", "accelerate", "datasets",
    )
except Exception as error:
    print(f"⚠️ 일부 파인튜닝 패키지를 자동 설치하지 못했습니다: {error}")


# QLoRA 파인튜닝 — 도메인 특화 모델 만들기

이 노트북은 Qwen 계열 모델을 QLoRA 방식으로 미세조정(fine-tuning)하는 과정을 학습용으로 정리한다. 핵심은 full fine-tuning처럼 모든 파라미터를 직접 업데이트하지 않고, 4bit 양자화(quantization)된 베이스 모델 위에 작은 LoRA adapter만 학습해 메모리 비용을 크게 줄인다는 점이다.

## 학습 목표
- QLoRA가 LoRA와 4bit 양자화를 어떻게 결합하는지 설명할 수 있다.
- 적은 양의 도메인 QA 데이터를 ChatML 형태로 만드는 흐름을 이해한다.
- `LoraConfig`, `BitsAndBytesConfig`, `SFTTrainer` 설정이 각각 무엇을 제어하는지 설명할 수 있다.
- 작은 데이터셋 파인튜닝의 효과와 한계를 균형 있게 말할 수 있다.


In [ ]:
from pathlib import Path
import importlib.util
import os
import sys

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import RuntimeConfig
from src.data_profiles import load_profile

runtime_config = RuntimeConfig.auto_detect()
profile_candidates = [
    ('tech_docs', PROJECT_ROOT / 'data' / 'eval' / 'eval_dataset_tech_docs.json'),
    ('korean_public', PROJECT_ROOT / 'data' / 'eval' / 'eval_dataset_korean.json'),
    ('demo', PROJECT_ROOT / 'data' / 'eval' / 'eval_dataset.json'),
]
selected_profile_name = next(name for name, path in profile_candidates if path.exists())
optional_modules = ['transformers', 'datasets', 'peft', 'trl', 'accelerate', 'bitsandbytes']
module_availability = {name: bool(importlib.util.find_spec(name)) for name in optional_modules}

if all(module_availability.values()):
    import torch
    from datasets import Dataset
    from peft import LoraConfig, PeftModel, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
    from trl import SFTTrainer
else:
    torch = None
    Dataset = None
    LoraConfig = None
    PeftModel = None
    get_peft_model = None
    AutoModelForCausalLM = None
    AutoTokenizer = None
    BitsAndBytesConfig = None
    TrainingArguments = None
    SFTTrainer = None

qlora_ready = all(module_availability.values()) and torch is not None and torch.cuda.is_available()
print(sys.executable)
{
    'selected_profile_name': selected_profile_name,
    'embedding_model': runtime_config.embedding_model,
    'llm_available': runtime_config.llm_available,
    'module_availability': module_availability,
    'cuda_available': bool(torch and torch.cuda.is_available()),
    'qlora_ready': qlora_ready,
}

## 구현: 학습 데이터 준비

**목적**
- 평가 데이터셋을 바로 파인튜닝 데이터의 출발점으로 바꾸는 과정을 이해한다.

**핵심 로직**
- 선택된 프로필의 eval dataset에서 `(question, retrieved context, gold answer)` 삼중항을 만든다.
- 이를 ChatML 스타일 한 문자열로 포맷해 SFT(supervised fine-tuning) 입력으로 사용한다.
- train/validation으로 나눠 과적합 여부를 최소한으로라도 볼 수 있게 한다.

**주요 파라미터**
- `selected_profile_name`: 현재 사용할 도메인 프로필
- `context`: retriever가 찾은 상위 3개 근거를 이어 붙인 문자열
- `text`: 최종 ChatML 형식 학습 샘플

데이터가 수십 개 수준일 때는 파인튜닝이 만능이 아니라는 점도 같이 기억해야 한다. 이런 규모에서는 프롬프트 엔지니어링이나 retrieval 개선이 더 큰 효과를 낼 수도 있다. 이 notebook은 QLoRA 구조를 배우는 실습으로 읽는 편이 맞다.

**결과 해석 가이드**
- `Train size`, `Val size`가 너무 작으면 loss 곡선이 안정적이어도 일반화는 제한적일 수 있다.
- `context`에 retrieved source가 붙는 것은 "도메인 QA" 학습 신호를 주기 위한 선택이다. 문맥 없는 QA보다 grounded style을 더 유도하기 쉽다.

### src 코드 펼침: `load_profile()`로 학습 재료 모으기

```python
def load_profile(name: str, backend: str = "tfidf", persist: bool = False) -> dict[str, Any]:
    materials = _load_profile_materials(name)
    runtime_config = RuntimeConfig.auto_detect()
    retriever = build_demo_index(
        raw_dir=materials["raw_dir"],
        persist=persist,
        backend=backend,
        recursive=materials["recursive"],
    )
    return {
        "name": name,
        "retriever": retriever,
        "eval_dataset": materials["eval_dataset"],
        "documents": materials["documents"],
        "chunks": materials["chunks"],
        "config": {...},
        "stats": stats,
    }
```

- 이 notebook에서는 별도 학습 전용 데이터 로더를 만들지 않고, 기존 profile 시스템을 재활용한다. 그래서 demo, tech_docs, korean_public 중 무엇을 학습 재료로 쓸지 한 줄로 바꿀 수 있다.
- `profile["eval_dataset"]`에서 `question`, `gold_answer`를 가져오고, `profile["chunks"]`에서 retrieval context를 뽑아 ChatML 학습 샘플을 만든다.
- 즉, "평가용 데이터셋을 학습용 초안 데이터로 재활용"하는 구조다. 데이터가 적은 교육용 실험에서는 빠르게 파인튜닝 파이프라인을 경험하기에 적합하다.
- 다만 실무에서는 evaluation leakage를 피해야 하므로, train/val/test를 명확히 분리하는 것이 더 중요하다.


In [ ]:
from sklearn.model_selection import train_test_split

training_profile = load_profile(selected_profile_name, persist=False)
raw_samples = training_profile['eval_dataset']


def format_chatml(question: str, context: str, answer: str) -> str:
    return (
        '<|system|>\n'
        'You are a domain QA assistant. Answer using the provided context only.\n'
        '<|user|>\n'
        f'Context:\n{context}\n\nQuestion: {question}\n'
        '<|assistant|>\n'
        f'{answer}'
    )


examples = []
for sample in raw_samples:
    retrieved = training_profile['retriever'].search(sample['question'], top_k=3)
    context = '\n\n'.join(
        f"[source={doc['source']}] {doc['text']}" for doc in retrieved
    ) or 'No retrieved context.'
    examples.append(
        {
            'question_id': sample['id'],
            'question_type': sample['question_type'],
            'question': sample['question'],
            'context': context,
            'answer': sample['gold_answer'],
            'text': format_chatml(sample['question'], context, sample['gold_answer']),
        }
    )

examples_frame = pd.DataFrame(examples)
train_frame, val_frame = train_test_split(examples_frame, test_size=0.2, random_state=42, shuffle=True)
train_frame = train_frame.reset_index(drop=True)
val_frame = val_frame.reset_index(drop=True)

if Dataset is not None:
    train_dataset = Dataset.from_pandas(train_frame[['text']], preserve_index=False)
    val_dataset = Dataset.from_pandas(val_frame[['text']], preserve_index=False)
else:
    train_dataset = None
    val_dataset = None

print(f'Selected profile: {selected_profile_name}')
print(f'Train size: {len(train_frame)}, Val size: {len(val_frame)}')
train_frame[['question_id', 'question_type', 'question', 'answer']].head()

## 구현: QLoRA 설정 이해하기

**목적**
        - 4bit 양자화와 LoRA adapter 설정이 실제 코드에서 어떤 형태로 들어가는지 본다.

        **핵심 로직**
        - `BitsAndBytesConfig`는 베이스 모델을 4bit로 로드해 VRAM 사용량을 줄인다.
        - `LoraConfig`는 작은 저랭크(rank) 행렬만 학습하게 만들어 fine-tuning 비용을 줄인다.
        - 둘을 합쳐 QLoRA 경로를 만든다.

        **실제 설정 코드: BitsAndBytesConfig**
        ```python
        bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)
        ```

        **실제 설정 코드: LoraConfig**
        ```python
        lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
        ```

        **코드 읽기 포인트**
        - `load_in_4bit=True`는 모델 파라미터를 4bit로 적재하겠다는 뜻이다. full precision보다 메모리를 크게 아낄 수 있다.
        - `bnb_4bit_quant_type='nf4'`는 QLoRA에서 자주 쓰는 quantization 타입이다.
        - `bnb_4bit_compute_dtype=torch.float16`은 저장은 4bit로 하되 계산은 fp16으로 하겠다는 뜻이다.
        - `r=16`은 LoRA rank다. 클수록 표현력은 늘지만 trainable parameter도 늘어난다.
        - `lora_alpha=32`는 보통 rank의 2배 수준으로 잡는 스케일링 값이다.
        - `target_modules=['q_proj', 'v_proj']`는 attention projection 일부에만 adapter를 꽂는 선택이다.
        - `lora_dropout=0.05`는 작은 데이터셋에서 과적합을 조금이라도 줄이기 위한 장치다.

        **결과 해석 가이드**
        - `model_bundle['ready']=False`면 현재 환경에서는 실제 학습이 불가능하므로, 이 notebook은 설정 해설 중심으로 읽으면 된다.
        - `trainable_params`가 매우 작게 나오면 QLoRA가 전체 모델이 아니라 adapter 일부만 학습하고 있다는 뜻이다.

        **💡 면접 포인트**
        - QLoRA는 full fine-tuning 대비 훨씬 적은 메모리로 도메인 적응을 시도할 수 있는 현실적인 선택지다.
        - rank, target_modules, quantization dtype은 품질과 비용을 함께 좌우하는 핵심 하이퍼파라미터다.

### src 코드 펼침: QLoRA 설정을 읽는 관점

```python
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
```

- `load_in_4bit=True`는 베이스 모델을 4bit로 줄여 VRAM 사용량을 크게 낮춘다. full fine-tuning이 아니라 QLoRA를 쓰는 핵심 이유다.
- `bnb_4bit_quant_type="nf4"`는 NormalFloat4 양자화 방식이다. 낮은 비트 수에서도 분포를 비교적 안정적으로 유지하도록 설계된 포맷이다.
- `bnb_4bit_compute_dtype=torch.float16`은 저장은 4bit로 하되 실제 계산은 fp16으로 하겠다는 뜻이다. 메모리와 연산 안정성의 절충점이다.
- `r=16`은 LoRA rank다. 클수록 표현력은 늘지만 adapter 파라미터 수도 같이 증가한다. 교육용 실험에서는 8~32 사이가 많이 쓰인다.
- `lora_alpha=32`는 adapter 업데이트의 스케일을 조정한다. 보통 `r`의 2배 근처에서 시작하는 경우가 많다.
- `target_modules`는 어떤 projection 레이어에 LoRA를 끼울지 지정한다. Qwen 계열에서는 `q_proj`, `k_proj`, `v_proj`, `o_proj`가 가장 흔한 출발점이다.
- `lora_dropout=0.05`는 작은 데이터셋에서 adapter가 과하게 암기하는 것을 조금 완화한다.
- DGX Spark 128GB 기준으로는 모델 크기와 sequence length에 따라 다르지만, 4B급 QLoRA라면 `per_device_train_batch_size=4`와 `gradient_accumulation_steps=4` 정도가 무난한 출발점이다. 길이가 길어지면 batch를 1~2로 낮추는 편이 안전하다.


In [ ]:
from pathlib import Path

adapter_output_dir = PROJECT_ROOT / 'artifacts' / 'finetuning' / 'qwen-qlora-demo'
model_id = os.getenv('FINETUNING_MODEL_ID', '').strip()
model_bundle = {
    'ready': False,
    'reason': '',
    'model_id': model_id or None,
    'trainable_params': None,
}
base_model = None
peft_model = None
tokenizer = None
bnb_config = None
lora_config = None


def parameter_summary(model) -> dict[str, int]:
    total = sum(parameter.numel() for parameter in model.parameters())
    trainable = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    return {'total': int(total), 'trainable': int(trainable)}


if not all(module_availability.values()):
    model_bundle['reason'] = 'Optional finetuning packages are missing. Run `uv sync --extra finetuning`.'
elif not qlora_ready:
    model_bundle['reason'] = 'QLoRA training requires CUDA + bitsandbytes. Current environment is CPU/MPS or missing GPU support.'
elif not model_id:
    model_bundle['reason'] = 'Set FINETUNING_MODEL_ID to a Hugging Face checkpoint or local path for a Qwen 4B-class model.'
else:
    try:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.float16,
        )
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        base_model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map='auto',
            trust_remote_code=True,
        )
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=['q_proj', 'v_proj'],
            lora_dropout=0.05,
            bias='none',
            task_type='CAUSAL_LM',
        )
        peft_model = get_peft_model(base_model, lora_config)
        model_bundle['ready'] = True
        model_bundle['trainable_params'] = parameter_summary(peft_model)
    except Exception as error:
        model_bundle['reason'] = str(error)
        base_model = None
        peft_model = None
        tokenizer = None

model_bundle

## 실험: 학습 실행

**목적**
        - 실제 supervised fine-tuning loop가 어떤 인자들로 구성되는지 이해한다.

        **핵심 로직**
        - `TrainingArguments`로 epoch, batch size, gradient accumulation, logging/eval/save 전략을 정의한다.
        - `SFTTrainer`가 text field를 읽어 causal LM 학습을 수행한다.
        - 환경이 준비되지 않았으면 안전하게 skip한다.

        **실제 설정 코드: TrainingArguments + SFTTrainer**
        ```python
        training_args = TrainingArguments(
    output_dir=str(adapter_output_dir),
    num_train_epochs=1,
    per_device_train_batch_size=per_device_batch_size,
    per_device_eval_batch_size=per_device_batch_size,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=2e-4,
    logging_steps=1,
    eval_strategy='epoch',
    save_strategy='epoch',
    fp16=True,
    report_to='none',
)
trainer = SFTTrainer(**trainer_kwargs)
        ```

        **주요 파라미터**
        - `per_device_train_batch_size=4`: DGX Spark 수준 GPU를 가정한 기본 마이크로 배치다.
        - `gradient_accumulation_steps=4`: 유효 배치 크기를 16으로 키우는 장치다.
        - `learning_rate=2e-4`: LoRA adapter 학습에 자주 쓰는 비교적 큰 학습률이다.
        - `max_seq_length=1024`: context + question + answer를 어느 정도까지 넣을지 결정한다.

        **코드 읽기 포인트**
        - `inspect.signature(SFTTrainer.__init__)`를 보는 이유는 `trl` 버전에 따라 `tokenizer` 대신 `processing_class`를 요구할 수 있기 때문이다.
        - skip 경로를 명시적으로 둔 것은 교육용 notebook이 CPU 환경에서도 끝까지 실행되게 만들기 위해서다.
        - `effective_batch_size = batch_size * gradient_accumulation`으로 해석하면 실제 업데이트 단위가 보인다.

        **결과 해석 가이드**
        - `status='trained'`면 실제 학습이 돌았다는 뜻이고, `skipped`면 환경 조건이 부족한 것이다.
        - 학습이 돌았더라도 데이터가 적으면 loss가 좋아 보여도 실제 generalization은 제한적일 수 있다.

        **💡 면접 포인트**
        - QLoRA 실험에서는 모델 품질만큼 환경 검증과 skip-safe 설계도 중요하다.
        - batch size와 gradient accumulation은 VRAM 제약 안에서 유효 배치를 확보하기 위한 실무적 타협점이다.


In [ ]:
import inspect

training_history_df = pd.DataFrame()
training_result = {'status': 'skipped', 'reason': model_bundle['reason'] or 'Training prerequisites not satisfied.'}
trainer = None
training_args = None

if model_bundle['ready'] and train_dataset is not None and val_dataset is not None:
    per_device_batch_size = 4
    gradient_accumulation_steps = 4
    training_args = TrainingArguments(
        output_dir=str(adapter_output_dir),
        num_train_epochs=1,
        per_device_train_batch_size=per_device_batch_size,
        per_device_eval_batch_size=per_device_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        learning_rate=2e-4,
        logging_steps=1,
        eval_strategy='epoch',
        save_strategy='epoch',
        fp16=True,
        report_to='none',
    )
    trainer_kwargs = {
        'model': peft_model,
        'args': training_args,
        'train_dataset': train_dataset,
        'eval_dataset': val_dataset,
        'dataset_text_field': 'text',
        'max_seq_length': 1024,
    }
    signature = inspect.signature(SFTTrainer.__init__)
    if 'processing_class' in signature.parameters:
        trainer_kwargs['processing_class'] = tokenizer
    elif 'tokenizer' in signature.parameters:
        trainer_kwargs['tokenizer'] = tokenizer

    trainer = SFTTrainer(**trainer_kwargs)
    train_output = trainer.train()
    trainer.save_model(str(adapter_output_dir))
    tokenizer.save_pretrained(str(adapter_output_dir))
    training_history_df = pd.DataFrame(trainer.state.log_history)
    training_result = {
        'status': 'trained',
        'global_step': getattr(train_output, 'global_step', None),
        'training_loss': getattr(train_output, 'training_loss', None),
        'output_dir': str(adapter_output_dir),
        'effective_batch_size': per_device_batch_size * gradient_accumulation_steps,
    }

training_result

## 결과 해석: 학습 모니터링

loss curve는 단순히 내려간다고 좋은 것이 아니라, train loss와 eval loss의 관계를 같이 읽어야 한다.

- train loss만 빠르게 내려가고 eval loss가 같이 좋아지지 않으면 과적합 가능성이 있다.
- 둘 다 완만하게 내려가다 평탄해지면 수렴 구간으로 볼 수 있다.
- loss history가 비어 있으면 현재 환경에서 training이 skip된 것이다. 이 경우 notebook은 설정 이해용으로 읽으면 된다.

작은 데이터셋에서는 예쁜 loss curve가 반드시 좋은 모델을 뜻하지 않는다는 점을 꼭 기억하자.


In [ ]:
loss_curve = pd.DataFrame()
if not training_history_df.empty:
    columns = [column for column in ['step', 'loss', 'eval_loss'] if column in training_history_df.columns]
    loss_curve = training_history_df[columns].copy()

if not loss_curve.empty and 'loss' in loss_curve.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(loss_curve['step'], loss_curve['loss'], marker='o', label='train_loss')
    if 'eval_loss' in loss_curve.columns and loss_curve['eval_loss'].notna().any():
        eval_rows = loss_curve.dropna(subset=['eval_loss'])
        ax.plot(eval_rows['step'], eval_rows['eval_loss'], marker='s', label='eval_loss')
    ax.set_title('QLoRA training loss curve')
    ax.set_xlabel('step')
    ax.set_ylabel('loss')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No loss history available. This usually means training was skipped in the current environment.')

loss_curve.head()

## 결과 해석: base vs fine-tuned

**목적**
- 같은 질문 10개에 대해 베이스 모델과 fine-tuned adapter가 어떤 차이를 보이는지 비교한다.

**핵심 로직**
- 같은 prompt를 base model과 tuned model에 넣고, 생성된 답변을 `score_answer()`로 비교한다.
- 여기서의 점수는 token overlap 기반이므로, 스타일보다 핵심 내용 회복 여부를 보는 용도다.

**주요 파라미터**
- `base_score`, `tuned_score`: gold answer 대비 정답 유사도
- `base_answer`, `tuned_answer`: 실제 생성 문장

**결과 해석 가이드**
- tuned score가 조금 올랐더라도, 데이터가 작으면 우연일 수 있으니 과해석하지 않는 편이 좋다.
- 특정 질문군에서만 개선됐다면, 그 도메인 표현에 과적합된 것일 수 있다.
- comparison table이 `skipped` 상태면 현재 환경에 adapter가 없거나 training을 실제로 돌리지 못한 것이다.

**💡 면접 포인트**
- 작은 데이터셋에서는 fine-tuning이 항상 retrieval 개선이나 prompt 개선보다 효율적이지 않을 수 있다.
- fine-tuning 효과는 평균 점수뿐 아니라 어떤 질문군에서 좋아졌는지까지 봐야 한다.

### src 코드 펼침: `score_answer()`

```python
def score_answer(predicted_answer: str, gold_answer: str, predicted_status: str, expected_status: str) -> float:
    if expected_status == "abstained":
        return 1.0 if predicted_status == "abstained" else 0.0
    if predicted_status == "abstained":
        return 0.0
    return token_f1(predicted_answer, gold_answer)
```

- fine-tuning 전후 비교에서도 같은 metric을 써야 공정하다. 그래서 여기서는 evaluation notebook과 동일한 `score_answer()`를 그대로 사용한다.
- 정답이 abstain이어야 하는 질문에 모델이 장황한 답을 내면 점수는 0이다. 즉, 많이 말한다고 무조건 좋은 모델이 아니다.
- 일반 answered 질문에서는 `token_f1()`를 사용한다. 표현이 조금 달라도 핵심 토큰이 많이 겹치면 부분 점수를 줄 수 있으므로, fine-tuning이 실제로 의미 있는 개선을 만들었는지 보기 좋다.

### src 코드 펼침: `SFTTrainer` 핵심 파라미터를 읽는 법

```python
trainer_args = {
    "max_seq_length": 1024,
    "num_train_epochs": 1,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
}
```

- `max_seq_length`는 한 샘플에서 모델이 보는 최대 토큰 길이다. context를 길게 넣을수록 retrieval 기반 QA에는 유리하지만, 메모리 사용량이 급격히 늘어난다.
- `num_train_epochs`는 전체 학습 데이터를 몇 번 반복할지 뜻한다. 데이터가 수십 개 수준이면 epoch를 많이 올릴수록 과적합 위험이 빠르게 커진다.
- `per_device_train_batch_size`는 GPU 한 장에 한 번에 올리는 샘플 수다. OOM이 나면 가장 먼저 줄일 값이다.
- `gradient_accumulation_steps`는 작은 배치를 여러 번 쌓아 큰 batch처럼 학습하는 방법이다. 예를 들어 batch 1에 accumulation 4면 체감상 batch 4처럼 동작한다.
- 즉, DGX에서는 VRAM 상황을 보며 `batch_size`와 `accumulation`을 함께 조절하는 것이 실전적인 튜닝 포인트다.


In [ ]:
from src.evaluator import score_answer

comparison_examples = examples_frame.head(10).copy()
comparison_df = pd.DataFrame()


def build_generation_prompt(row: pd.Series) -> str:
    return (
        'You are a domain QA assistant. Use the context only.\n\n'
        f"Context:\n{row['context']}\n\nQuestion: {row['question']}\nAnswer:"
    )


def generate_answer(model, tokenizer, prompt: str, max_new_tokens: int = 128) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return decoded.split('Answer:', 1)[-1].strip()


if model_bundle['ready'] and adapter_output_dir.exists():
    try:
        base_eval_model = AutoModelForCausalLM.from_pretrained(
            model_bundle['model_id'],
            quantization_config=bnb_config,
            device_map='auto',
            trust_remote_code=True,
        )
        tuned_eval_model = PeftModel.from_pretrained(base_eval_model, str(adapter_output_dir))
        rows = []
        for row in comparison_examples.to_dict(orient='records'):
            prompt = build_generation_prompt(pd.Series(row))
            with tuned_eval_model.disable_adapter():
                base_answer = generate_answer(tuned_eval_model, tokenizer, prompt)
            tuned_answer = generate_answer(tuned_eval_model, tokenizer, prompt)
            rows.append(
                {
                    'question_id': row['question_id'],
                    'question': row['question'],
                    'gold_answer': row['answer'],
                    'base_answer': base_answer,
                    'tuned_answer': tuned_answer,
                    'base_score': score_answer(base_answer, row['answer'], 'answered', 'answered'),
                    'tuned_score': score_answer(tuned_answer, row['answer'], 'answered', 'answered'),
                }
            )
        comparison_df = pd.DataFrame(rows)
    except Exception as error:
        comparison_df = pd.DataFrame([{'status': 'skipped', 'reason': str(error)}])
else:
    comparison_df = pd.DataFrame([{'status': 'skipped', 'reason': model_bundle['reason'] or 'Fine-tuned adapter unavailable.'}])

comparison_df.head(10)

## 핵심 정리

이 노트북을 통해 QLoRA는 적은 메모리로 도메인 특화 적응을 시도할 수 있는 실용적 방법이라는 점을 확인했다. `BitsAndBytesConfig`로 베이스 모델을 4bit로 줄이고, `LoraConfig`로 adapter 일부만 학습하며, `SFTTrainer`로 ChatML 형식 데이터를 supervised fine-tuning한다. 다만 데이터가 수십 개 수준이면 파인튜닝 효과가 제한적일 수 있고, retrieval 개선이나 prompt 설계가 더 큰 개선을 줄 수도 있다.

**💡 면접 포인트**
- QLoRA로 도메인 특화 모델을 만들 수 있지만, 작은 데이터셋에서는 과적합과 효과 제한을 항상 같이 봐야 한다.
- full fine-tuning보다 훨씬 적은 VRAM으로 adapter 학습이 가능하다는 점이 QLoRA의 실무적 장점이다.
- LoRA rank, target module, batch/accumulation 설정이 품질과 비용에 미치는 영향을 실험적으로 설명할 수 있다.
